# W03 — Data Contract (Foundations)

**Lane:** content-performance forecasting on the FlyRank warehouse — *rank a client's content by the clicks it will earn in the next 7 days.*

**Flow of this notebook**

```
Colab Secret HF_TOKEN → Hugging Face login → FlyRank/internship-warehouse (gated) → DuckDB over the parquet files → this notebook
```

| Part | What | Where |
|---|---|---|
| 0 | Setup & auth (token comes from the Colab Secret — never pasted in a cell) | §0 |
| 1 | **The contract** — five plain-words answers | §1 |
| 2 | **Prove three facts** with three small queries on a mid-panel month | §2 |
| 3 | **Five features max**, each "knowable at the decision moment because…" | §3 |
| 4 | **The trap** — add one label-derived column, watch the score jump, delete it | §4 |
| 5 | **Self-check** | §5 |

> **Before running:** (1) request access on the dataset page and wait for approval, (2) Colab → 🔑 *Secrets* → add `HF_TOKEN` and switch **Notebook access ON**, (3) *Runtime → Run all*.
> The final month (June 2026) is a sealed test month; everything here uses a mid-panel month.

## 0 · Setup & authentication

In [ ]:
!pip -q install duckdb huggingface_hub

In [ ]:
# --- Authentication: token is read from the Colab Secret named HF_TOKEN ---
import os

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")          # Colab Secret (🔑 panel on the left)
except ImportError:                              # running outside Colab -> environment variable
    HF_TOKEN = os.environ.get("HF_TOKEN")
except Exception as e:                           # secret missing / notebook access switched off
    raise RuntimeError(
        "Could not read the Colab Secret HF_TOKEN. Open the 🔑 Secrets panel, add HF_TOKEN, "
        "and toggle 'Notebook access' ON."
    ) from e

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN is empty. Add it as a Colab Secret (never paste it into a cell — the repo is public).")

from huggingface_hub import login, HfApi
login(token=HF_TOKEN, add_to_git_credential=False)
print("Authenticated as:", HfApi().whoami()["name"])

In [ ]:
# --- Configuration (edit here, nowhere else) ---
from datetime import date, timedelta

REPO_ID     = "FlyRank/internship-warehouse"
DIM_CLIENTS = "dim_clients"
FACT        = "fact_content_daily_performance"

MONTH        = "2026-03"      # mid-panel month.  NEVER use the final month (2026-06): it is the sealed test month.
DECISION_DAY = 15             # decision moment = the morning of MONTH-15

# Column overrides for the fact table. Leave None to auto-detect from the schema (detection is printed below).
COLS = dict(client=None, content=None, date=None, clicks=None, impressions=None, position=None)
EXTRA_GRAIN_KEYS = []          # if Query 1 shows duplicates, list extra key columns here (e.g. ["device"])
CLIENT_CREATED_COL = None      # dim_clients "client_crea…" column; None = auto-detect

# Derived dates -----------------------------------------------------------------
y, m = map(int, MONTH.split("-"))
MONTH_START = date(y, m, 1)
NEXT_MONTH_START = date(y + (m == 12), (m % 12) + 1, 1)
D          = date(y, m, DECISION_DAY)               # decision moment
PRE_START  = D - timedelta(days=7);  PRE_END  = D - timedelta(days=1)    # last 7 days before decision
PRIOR_START= D - timedelta(days=14); PRIOR_END= D - timedelta(days=8)    # the 7 days before that
NEXT_START = D;                      NEXT_END = D + timedelta(days=6)    # label window
assert PRIOR_START >= MONTH_START and NEXT_END < NEXT_MONTH_START, "windows must fit inside the chosen month"
assert MONTH != "2026-06", "2026-06 is the sealed test month"
print(f"decision moment {D} | features {PRIOR_START}..{PRE_END} | label window {NEXT_START}..{NEXT_END}")

In [ ]:
# --- Discover the parquet files behind the two tables we need, then open a DuckDB connection ---
import re, duckdb, pandas as pd
from huggingface_hub.utils import GatedRepoError, HfHubHTTPError

api = HfApi()
try:
    all_files = api.list_repo_files(REPO_ID, repo_type="dataset")
except GatedRepoError:
    raise SystemExit("403 / gated: open https://huggingface.co/datasets/FlyRank/internship-warehouse and press 'request access', then re-run.")
except HfHubHTTPError as e:
    raise SystemExit(f"Hugging Face error: {e}\nIf this is a 403, check that access was approved and that HF_TOKEN is a plain *Read* token.")

parquet_files = [f for f in all_files if f.endswith(".parquet")]

def table_urls(name):
    """hf:// URLs of the parquet files for one table (exact folder / file-prefix match, so '_sample' tables are NOT picked up)."""
    pat = re.compile(rf"(^|/){re.escape(name)}(/|-|\.|$)")
    return [f"hf://datasets/{REPO_ID}/{f}" for f in parquet_files if pat.search(f)]

TABLE_URLS = {t: table_urls(t) for t in (DIM_CLIENTS, FACT)}
for t, urls in TABLE_URLS.items():
    print(f"{t}: {len(urls)} parquet file(s)")
    if not urls:
        print("  none matched. Parquet files in the repo:", *parquet_files[:40], sep="\n   ")
        raise SystemExit(f"Could not locate files for {t}; set TABLE_URLS[{t!r}] by hand.")

# If the fact table is hive-partitioned by month (…/month=2026-03/…), read only that month's files.
_m = [u for u in TABLE_URLS[FACT] if f"month={MONTH}" in u]
if _m:
    TABLE_URLS[FACT] = _m
    print(f"  -> partition pruning: {len(_m)} file(s) for month={MONTH}")

def rp(urls):
    """SQL fragment: read_parquet over a list of URLs."""
    return "read_parquet([" + ", ".join("'" + u + "'" for u in urls) + "])"

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
try:                                    # native Hugging Face support in DuckDB (token never printed)
    con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE HUGGINGFACE, TOKEN '{HF_TOKEN}')")
    con.sql(f"SELECT * FROM {rp(TABLE_URLS[DIM_CLIENTS])} LIMIT 1").fetchall()
    print("DuckDB reads hf:// natively.")
except Exception as e:                  # fallback: fsspec filesystem from huggingface_hub
    print("Native hf:// failed (", type(e).__name__, ") -> falling back to HfFileSystem")
    from huggingface_hub import HfFileSystem
    con.register_filesystem(HfFileSystem(token=HF_TOKEN))
    con.sql(f"SELECT * FROM {rp(TABLE_URLS[DIM_CLIENTS])} LIMIT 1").fetchall()

In [ ]:
# --- Load dim_clients (104 rows) and inspect the fact schema; auto-detect the columns we need ---
dim_urls  = TABLE_URLS[DIM_CLIENTS]
dim_cols  = con.sql(f"DESCRIBE SELECT * FROM {rp(dim_urls)}").df()["column_name"].tolist()
required  = ["client_hash_id", "is_active", "has_gsc_access", "has_ga4_access", "access_profile"]
missing   = [c for c in required if c not in dim_cols]
assert not missing, f"dim_clients is missing {missing}; columns are {dim_cols}"

created_col = CLIENT_CREATED_COL or next((c for c in dim_cols if c.lower().startswith("client_creat")), None)
assert created_col, f"No 'client_crea…' column found in {dim_cols}; set CLIENT_CREATED_COL."

con.execute(f"""
    CREATE OR REPLACE TABLE dim_clients AS
    SELECT * EXCLUDE ("{created_col}"), CAST("{created_col}" AS DATE) AS client_created
    FROM {rp(dim_urls)}
""")
display(con.sql("SELECT * FROM dim_clients LIMIT 5").df())

fact_schema = con.sql(f"DESCRIBE SELECT * FROM {rp(TABLE_URLS[FACT])}").df()
fact_cols   = fact_schema["column_name"].tolist()
print("\nFact table schema:")
display(fact_schema[["column_name", "column_type"]])

def pick(what, override, candidates, keyword):
    if override:
        if override not in fact_cols:
            raise KeyError(f"{what}: '{override}' not in {fact_cols}")
        return override
    lower = {c.lower(): c for c in fact_cols}
    for cand in candidates:
        if cand in lower:
            return lower[cand]
    hits = [c for c in fact_cols if keyword in c.lower()]
    if hits:
        return hits[0]
    raise KeyError(f"Could not auto-detect the {what} column. Columns: {fact_cols}. Set it in COLS.")

C_CLIENT = pick("client",      COLS["client"],      ["client_hash_id", "client_id"], "client")
C_CONTENT= pick("content",     COLS["content"],     ["content_id", "content_hash_id", "page_id", "url_hash", "content_key"], "content")
C_CLICKS = pick("clicks",      COLS["clicks"],      ["clicks", "total_clicks", "gsc_clicks"], "click")
C_IMPR   = pick("impressions", COLS["impressions"], ["impressions", "total_impressions", "gsc_impressions"], "impression")
C_POS    = pick("position",    COLS["position"],    ["avg_position", "position", "average_position", "gsc_position"], "position")

date_like = fact_schema[fact_schema["column_type"].str.upper().str.startswith(("DATE", "TIMESTAMP"))]["column_name"].tolist()
_lower = {c.lower(): c for c in fact_cols}
C_DATE = (COLS["date"]
          or next((_lower[c] for c in ["date", "event_date", "report_date", "metric_date", "day", "dt"] if c in _lower), None)
          or (date_like[0] if date_like else None))
assert C_DATE, f"Could not detect the date column in {fact_cols}; set COLS['date']."

print("Detected columns ->", dict(client=C_CLIENT, content=C_CONTENT, date=C_DATE, clicks=C_CLICKS, impressions=C_IMPR, position=C_POS))

In [ ]:
# --- Pull ONE mid-panel month of the fact table into a local DuckDB table (the only heavy read; may take a few minutes) ---
extra = "".join(f', "{c}"' for c in EXTRA_GRAIN_KEYS)
con.execute(f"""
    CREATE OR REPLACE TABLE fact_month AS
    SELECT "{C_CLIENT}"  AS client_id,
           "{C_CONTENT}" AS content_id,
           CAST("{C_DATE}" AS DATE) AS d,
           "{C_CLICKS}"  AS clicks,
           "{C_IMPR}"    AS impressions,
           "{C_POS}"     AS position
           {extra}
    FROM {rp(TABLE_URLS[FACT])}
    WHERE CAST("{C_DATE}" AS DATE) >= DATE '{MONTH_START}'
      AND CAST("{C_DATE}" AS DATE) <  DATE '{NEXT_MONTH_START}'
""")
print("fact_month rows:", con.sql("SELECT COUNT(*) FROM fact_month").fetchone()[0])

## 1 · The contract — five answers in plain words

Edit the text if your lane differs. The self-check in §5 refuses to pass while any answer still contains the word *TODO*.

In [ ]:
from IPython.display import Markdown, display

CONTRACT = {
 "1. What one row means": (
    f"In the raw fact table one row = one client x one content item x one calendar day of Search Console performance "
    f"(this claim is tested in Query 1). In my modelling frame one row = one content item of one client at ONE decision moment "
    f"({D}, morning)."),
 "2. Tables I use": (
    f"`{DIM_CLIENTS}` (who the client is: access flags, created date) and `{FACT}` (daily clicks, impressions, position per content item). "
    f"I do not use dim_content or fact_content_query_90d."),
 "3. Time window": (
    f"Features may only look at dates {PRIOR_START} to {PRE_END} (two 7-day blocks before the decision moment). "
    f"The label looks at {NEXT_START} to {NEXT_END}. Month = {MONTH}; the final month (June 2026) stays sealed as the test month."),
 "4. What I predict / rank": (
    f"Label: total clicks a content item gets in the 7 days from {NEXT_START} (`clicks_next7`). "
    f"I rank content by predicted clicks and score the ranking with Spearman correlation on held-out clients."),
 "5. What I deliberately exclude": (
    f"Anything dated {D} or later inside a feature (next-7-day clicks, impressions, position, or anything computed from them); "
    f"the June 2026 test month; and the `is_active` flag as a FEATURE (it is today's snapshot, not what was true in {MONTH})."),
}
for k, v in CONTRACT.items():
    display(Markdown(f"**{k}** — {v}"))

## 2 · Prove three facts — three small queries on `month=` the chosen month

Everything below runs on the single mid-panel month loaded above (never the final month).

**Fact 1 – the grain:** is one row really one client × content × day?

In [ ]:
# QUERY 1 — grain: rows vs distinct (client, content, day [, extra keys])
keys = "client_id, content_id, d" + "".join(f', "{c}"' for c in EXTRA_GRAIN_KEYS)
Q1_SQL = f"""
SELECT COUNT(*)                                             AS n_rows,
       (SELECT COUNT(*) FROM (SELECT DISTINCT {keys} FROM fact_month)) AS n_distinct_grain
FROM fact_month
"""
Q1_df = con.sql(Q1_SQL).df()
Q1_df["duplicate_rows"] = Q1_df["n_rows"] - Q1_df["n_distinct_grain"]
display(Q1_df)
print("✅ Grain holds: one row = one client x content x day." if Q1_df.loc[0, "duplicate_rows"] == 0 else
      "⚠️ Grain is NOT unique: another key column exists. Inspect the fact schema and add it to EXTRA_GRAIN_KEYS, then re-run.")

**Fact 2 – row count and date span** of my slice (the chosen month, all clients):

In [ ]:
# QUERY 2 — slice size and date span
Q2_SQL = """
SELECT COUNT(*)                     AS n_rows,
       MIN(d)                       AS first_date,
       MAX(d)                       AS last_date,
       COUNT(DISTINCT d)            AS n_days,
       COUNT(DISTINCT client_id)    AS n_clients,
       COUNT(DISTINCT content_id)   AS n_content
FROM fact_month
"""
Q2_df = con.sql(Q2_SQL).df()
display(Q2_df)

**Fact 3 – availability, checked with `IS TRUE`.**
The flags in `dim_clients` are booleans that can be `NULL`. `= TRUE` and bare `WHERE flag` silently drop `NULL`s *and*, when negated (`NOT flag`), also drop them from the "false" side. `IS TRUE` / `IS NOT TRUE` treat `NULL` explicitly, so the counts add up.

In [ ]:
# QUERY 3 — availability: how many rows / clients survive  is_active IS TRUE AND has_gsc_access IS TRUE
Q3_SQL = """
SELECT COUNT(*)                                                                               AS rows_total,
       COUNT(*) FILTER (WHERE c.is_active IS TRUE AND c.has_gsc_access IS TRUE)               AS rows_available,
       COUNT(DISTINCT f.client_id)                                                            AS clients_total,
       COUNT(DISTINCT f.client_id) FILTER (WHERE c.is_active IS TRUE AND c.has_gsc_access IS TRUE) AS clients_available
FROM fact_month f
LEFT JOIN dim_clients c ON f.client_id = c.client_hash_id
"""
Q3_df = con.sql(Q3_SQL).df()
Q3_df["pct_rows_surviving"] = (100 * Q3_df["rows_available"] / Q3_df["rows_total"]).round(1)
display(Q3_df)

# context: how many NULL flags exist in dim_clients (why IS TRUE matters)
display(con.sql("""
    SELECT COUNT(*) AS clients,
           COUNT(*) FILTER (WHERE is_active IS NULL)        AS is_active_null,
           COUNT(*) FILTER (WHERE has_gsc_access IS NULL)   AS gsc_null,
           COUNT(*) FILTER (WHERE is_active IS TRUE AND has_gsc_access IS TRUE) AS available_now
    FROM dim_clients""").df())

## 3 · The feature frame — five features, each "knowable at the decision moment because…"

*Population:* content items of available clients that had **impressions in the 7 days before the decision moment** (that is something you can know on the morning of the decision).
All feature windows end the day **before** the decision day.

In [ ]:
# 'Available when?' line for every feature — the contract for this frame
FEATURES = {
 "clicks_7d_pre": (
    f"Sum of clicks on {PRE_START}..{PRE_END}. Knowable at the decision moment because every one of those days is already "
    f"in the past (the daily sync has landed the previous day's data by the morning of {D})."),
 "impressions_7d_pre": (
    f"Sum of impressions on {PRE_START}..{PRE_END}. Knowable at the decision moment because it uses the same fully-elapsed days as clicks_7d_pre."),
 "avg_position_7d_pre": (
    f"Impression-weighted average Search Console position on {PRE_START}..{PRE_END}. Knowable at the decision moment because it only uses elapsed days."),
 "clicks_momentum": (
    f"clicks in {PRE_START}..{PRE_END} minus clicks in {PRIOR_START}..{PRIOR_END}. Knowable at the decision moment because both blocks end before {D}."),
 "client_age_days": (
    f"Days between the client's created date and {D}. Knowable at the decision moment because the created date never changes after sign-up."),
}

frame_sql = f"""
WITH base AS (
    SELECT f.client_id, f.content_id, f.d, f.clicks, f.impressions, f.position, c.client_created
    FROM fact_month f
    JOIN dim_clients c ON f.client_id = c.client_hash_id
    WHERE c.is_active IS TRUE
      AND c.has_gsc_access IS TRUE
      AND (c.client_created IS NULL OR c.client_created <= DATE '{D}')
)
SELECT client_id, content_id,
  COALESCE(SUM(clicks)      FILTER (WHERE d BETWEEN DATE '{PRE_START}' AND DATE '{PRE_END}'), 0)   AS clicks_7d_pre,
  COALESCE(SUM(impressions) FILTER (WHERE d BETWEEN DATE '{PRE_START}' AND DATE '{PRE_END}'), 0)   AS impressions_7d_pre,
  SUM(position * impressions) FILTER (WHERE d BETWEEN DATE '{PRE_START}' AND DATE '{PRE_END}')
     / NULLIF(SUM(impressions) FILTER (WHERE d BETWEEN DATE '{PRE_START}' AND DATE '{PRE_END}'), 0) AS avg_position_7d_pre,
  COALESCE(SUM(clicks) FILTER (WHERE d BETWEEN DATE '{PRE_START}'   AND DATE '{PRE_END}'),   0)
- COALESCE(SUM(clicks) FILTER (WHERE d BETWEEN DATE '{PRIOR_START}' AND DATE '{PRIOR_END}'), 0)   AS clicks_momentum,
  DATE_DIFF('day', MAX(client_created), DATE '{D}')                                               AS client_age_days,
  -- label + helper for the leakage experiment (AFTER the decision moment: never a feature)
  COALESCE(SUM(clicks) FILTER (WHERE d BETWEEN DATE '{NEXT_START}' AND DATE '{NEXT_END}'), 0)     AS clicks_next7,
  COUNT(DISTINCT d)    FILTER (WHERE d BETWEEN DATE '{NEXT_START}' AND DATE '{NEXT_END}')         AS days_next7
FROM base
GROUP BY client_id, content_id
HAVING COALESCE(SUM(impressions) FILTER (WHERE d BETWEEN DATE '{PRE_START}' AND DATE '{PRE_END}'), 0) > 0
"""
frame = con.sql(frame_sql).df()
print(f"frame: {len(frame):,} rows | {frame['client_id'].nunique()} clients | features = {list(FEATURES)}")
display(frame[["client_id", "content_id", *FEATURES, "clicks_next7"]].head(8))

# guard: every feature window must end strictly before the decision day
assert PRE_END < D and PRIOR_END < D, "a feature window reaches the decision day"
assert len(FEATURES) <= 5, "five features, max"

In [ ]:
# The honest "available when?" table
display(Markdown("| feature | available when? |\n|---|---|\n" + "\n".join(f"| `{k}` | {v} |" for k, v in FEATURES.items())))

## 4 · The trap — one label-derived column, on purpose

Recipe: (1) score the honest 5 features, (2) add **one** column that secretly comes from the label window, (3) watch the quick score jump toward perfect, (4) delete it and keep the honest number.
The trap column is `clicks_per_active_day_next7 = clicks_next7 / days_next7` — it *looks* like a harmless "daily click rate", but it is computed from the very days we are trying to predict.

In [ ]:
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import GroupShuffleSplit
from scipy.stats import spearmanr

# quick, reproducible scorer: holdout by CLIENT, target = log1p(clicks_next7), score = Spearman rank correlation
qs = frame.sample(n=min(len(frame), 300_000), random_state=0).reset_index(drop=True)
y  = np.log1p(qs["clicks_next7"])
train_idx, test_idx = next(GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42).split(qs, groups=qs["client_id"]))

def quick_score(cols):
    model = HistGradientBoostingRegressor(max_iter=100, random_state=42)
    model.fit(qs.loc[train_idx, cols], y.iloc[train_idx])
    return spearmanr(model.predict(qs.loc[test_idx, cols]), y.iloc[test_idx])[0]

FEATURE_COLS = list(FEATURES)
SCORE_HONEST = quick_score(FEATURE_COLS)
print(f"honest score (5 features)               : {SCORE_HONEST:.3f}")

# --- add the label-derived column on purpose ---
LEAK_COL = "clicks_per_active_day_next7"
qs[LEAK_COL] = np.where(qs["days_next7"] > 0, qs["clicks_next7"] / qs["days_next7"].clip(lower=1), 0.0)
SCORE_LEAK = quick_score(FEATURE_COLS + [LEAK_COL])
print(f"score WITH the leaky column             : {SCORE_LEAK:.3f}   <-- suspiciously high = leakage")

# --- delete it, keep the honest number ---
qs = qs.drop(columns=[LEAK_COL])
SCORE_AFTER = quick_score(FEATURE_COLS)
print(f"score after deleting the leaky column   : {SCORE_AFTER:.3f}   <-- the number I keep")
assert LEAK_COL not in qs.columns and abs(SCORE_AFTER - SCORE_HONEST) < 1e-9

display(pd.DataFrame({"experiment": ["honest 5 features", "+ label-derived column", "leaky column deleted"],
                      "spearman": [SCORE_HONEST, SCORE_LEAK, SCORE_AFTER]}).round(3))

**Reading:** a jump like this from a single column is not "a better model" — it is the future leaking into the features. The honest number above is the only one that goes in the write-up.

In [ ]:
LIMITATION = (
    "Survivorship bias in the slice: I select clients using TODAY's flags (is_active, has_gsc_access), so clients that churned or lost "
    f"Search Console access after {MONTH} are missing. The model only learns from clients that stayed healthy, and I evaluate on a single "
    "mid-panel month, so seasonality in that month is baked into the score."
)
display(Markdown(f"**Named limitation of my slice:** {LIMITATION}"))

## 5 · Self-check

In [ ]:
checks = []
def check(name, ok): checks.append((name, bool(ok)))

check("Contract: five plain-words answers, none left as TODO",
      len(CONTRACT) == 5 and all(v.strip() and "TODO" not in v for v in CONTRACT.values()))
check("Exactly three verification queries ran (grain / row count + span / availability)",
      all(x is not None for x in (Q1_df, Q2_df, Q3_df)))
check("Grain query shows zero duplicate rows (or EXTRA_GRAIN_KEYS documents the fix)",
      int(Q1_df.loc[0, "duplicate_rows"]) == 0 or len(EXTRA_GRAIN_KEYS) > 0)
check("Availability was checked with IS TRUE", "IS TRUE" in Q3_SQL.upper() and "IS TRUE" in frame_sql.upper())
check("Queries ran on a mid-panel month, not the sealed final month", MONTH != "2026-06")
check("At most five features", len(FEATURES) <= 5)
check("Every feature has an 'available when?' line", all(v.strip() and "TODO" not in v for v in FEATURES.values()))
check("Feature windows end before the decision day", PRE_END < D and PRIOR_END < D)
check("Leak experiment shown: score jumped with the trap column, then it was removed",
      SCORE_LEAK > SCORE_HONEST and LEAK_COL not in qs.columns and LEAK_COL not in FEATURES)
check("No label / label-window column among the features",
      not any(c.endswith("_next7") for c in FEATURES))
check("One named limitation written", LIMITATION.strip() and "TODO" not in LIMITATION)

for name, ok in checks:
    print("✅" if ok else "❌", name)
print("\nALL CHECKS PASSED — save, commit with outputs visible, and submit the repo URL." if all(ok for _, ok in checks)
      else "\nFix the ❌ items above and re-run this cell.")

### Done-looks-like checklist
- [x] five plain-words contract answers (§1)
- [x] exactly three verification queries with outputs visible; availability via `IS TRUE` (§2)
- [x] five-feature frame with an "available when?" line per feature (§3)
- [x] deliberate-leak experiment shown, then removed (§4)
- [x] one named limitation (§4)

**Commit this notebook, executed, as `work/notebooks/w03_data_contract.ipynb`, then submit the repo URL.**